# INFO284 Machine Learning Exam - Spring 2026

---

# Task 1: Sentiment Analysis

In this notebook we build a pipeline for predicting review scores from review text. After exploring and cleaning the data, we train and evaluate four models:

1. **Logistic Regression**
2. **Support Vector Machine (LinearSVC)**
3. **LightGBM**
4. **Bidirectional LSTM**

All four share the same cleaned text and 80/20 train/test split for a fair comparison, but each applies its own feature extraction on top. We tune each model with cross-validated hyperparameter search within practical time and compute limits, and use class weighting throughout to handle the dataset's strong class imbalance.

## 3. Exploratory Data Analysis

We start by understanding the rating distribution, review lengths, and word frequencies. This informs our preprocessing choices and helps anticipate challenges like class imbalance.

### 3.1 Rating Distribution

The dataset is heavily imbalanced: rating 5 makes up almost half of all reviews, while ratings 2 and 3 have very few samples. A model that always predicts 5 would already achieve nearly 50% accuracy without learning anything meaningful. We address this with class weighting, text augmentation (LSTM only), and by using macro-averaged F1 as the primary metric.

### 3.2 Review Length

Review length is right-skewed: most reviews are short, with a long tail of detailed ones. 1-star reviews tend to be longer (more detailed complaints), while 5-star reviews cluster at shorter lengths. Length carries some signal, but we leave it out as a feature and let the models learn from text directly. This analysis also informs the LSTM's `MAX_LEN=100`: most reviews fall under this token count after stopword removal, so truncation only affects a small fraction of samples.

### 3.3 Word and Bigram Frequencies

The most frequent unigrams include sentiment-laden words ("love", "cant", "worst") but also domain noise like "app" and "whatsapp", which appear often without carrying sentiment, we therefore add these to the stopword list. The same analysis motivates keeping negation words ("not", "no", "never") in, since they appear frequently and flip surrounding sentiment. The bigram analysis reveals nonsensical repeating sequences in many reviews, which we address in the cleaning pipeline below.

## 4. Data Quality Checks and Cleaning

Before training we identify and remove problematic entries: gibberish (random characters), spam (heavily repeated words), non-English text, and emoji-only reviews. These would add noise without contributing useful signal.

## 5. Shared Text Preprocessing

All four models share the same cleaning pipeline:

- **Lowercasing** - so "Good" and "good" are treated identically.
- **Remove URLs and emails** - they carry no sentiment signal.
- **Remove non-letter characters** - punctuation and digits add noise for our bag-of-words models.
- **Remove stopwords (with exceptions)**
- **Normalise whitespace** - for consistent tokenization.

Each model then applies its own feature extraction (TF-IDF variants, or integer tokenization for the LSTM) on top of this cleaned text.

## 6. Target Variable and Train/Test Split

The target for all four models is the `rating` column (1–5 stars), directly addressing the task of predicting the assigned review score. The 80/20 train/test split is stratified on `rating` to preserve the class distribution in both sets. With `random_state=42` and a single split, all four models are trained and evaluated on identical data.

## 7. Shared Design Decisions

To avoid repeating the same justifications in every model discussion, we summarize the cross-model decisions here:

- **Class weighting** (`class_weight='balanced'` or equivalent) is applied to all four models. This is a deliberate trade-off: accuracy typically drops because the model no longer leans on the majority class, but macro F1 improves and all five classes are predicted rather than just rating 5.
- **Macro F1 as primary metric.** Accuracy is dominated by rating 5 due to imbalance. The macro F1 treats all classes equally and better reflects performance on the harder minority classes.
- **Hyperparameter tuning via cross-validation** on the training set only, with the test set held out untouched until the final evaluation.
- **Why these models.** Logistic Regression and LinearSVC are strong baselines for high-dimensional sparse text and scale well to large vocabularies. LightGBM tests whether non-linear interactions help. The LSTM tests whether sequential context outperforms bag-of-words on this dataset size.

The following sections only discuss what is unique to each model.

## 8. Model 1: Logistic Regression

**Design choices.** TF-IDF with `max_features=5000`, bigrams, and `min_df=5` for a compact feature space. Grid search over `C` and `penalty` (8 combinations, 3-fold CV) optimizing macro F1. `solver='saga'` supports both L1 and L2 with multiclass.

### Results and discussion

The grid search selected `C=1.0` with L2 penalty (CV macro F1: 0.330). Lower C values underfit the sparse TF-IDF features, higher values gave no gain.

The model reaches 49.4% accuracy and macro F1 of 0.33. Per-class results are reasonably even: ratings 2–4 reach F1 of 0.12–0.17, rating 5 has recall 0.61, and rating 1 is the strongest minority class with macro F1 = 0.55, helped by its distinctive negative language.

The main limitation is the 5,000-feature TF-IDF vocabulary, which likely caps how much signal the model can extract from the harder middle ratings. This is a limitation the SVM section explores.

## 9. Model 2: Support Vector Machine (LinearSVC)

**Design choices.** TF-IDF with `max_features=15,000`, bigrams, `sublinear_tf=True`, and `min_df=2`. This is a larger feature space than the LR-model to give the SVM-model more discriminative signal. C tuned with 5-fold CV over [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0] using macro F1.

### Results and discussion

CV selects `C=0.1` (CV macro F1: 0.335), with scores declining steadily for higher C. The default `C=1.0` overfits the TF-IDF space, and stronger regularization generalizes better. The tuned model reaches 60.1% accuracy and macro F1 of 0.35, up from 55.4% and 0.32 untuned.

Per-class spread is the widest of the four models. Rating 5 is strongest (F1 = 0.77) and rating 1 reaches 0.61, while ratings 2/3/4 sit at 0.11, 0.07, and 0.16 respectively. Tuning lifts rating 1 recall to 0.69, but ratings 2/3/4 stay at 0.06–0.13. Stronger regularization helps the easiest minority class without rescuing the highly ambiguous middle ratings.

Feature importances explain the middle-class weakness: ratings 1 and 5 have sharp sentiment cues ("worst", "bad" vs "love", "great"), while 2/3/4 show ambiguous signals ("difficult", "somewhat", "good communication") which would be hard to resolve by the tuning itself.

## 10. Model 3: LightGBM

**Design choices.** TF-IDF with `max_features=20,000`, trigrams, `sublinear_tf=True`, and `max_df=0.9` to filter both rare and overly common terms. We initially ran a grid search over `n_estimators`, `learning_rate`, and `num_leaves` (8 combinations, 3-fold CV). To check whether a richer hyperparameter space would help, we then ran a randomized search over seven hyperparameters, adding `reg_alpha`, `reg_lambda`, `min_child_samples`, and `colsample_bytree` (20 sampled combinations, 3-fold CV). These four are the parameters most relevant for sparse high-dimensional features: L1/L2 regularization controls overfitting on the many near-zero TF-IDF weights, `min_child_samples` lets trees capture rare minority-class patterns, and column subsampling regularizes splits on the wide feature matrix. The expanded search roughly doubled runtime, but improved macro F1 by less than 0.01, so we kept the smaller grid.

### Results and discussion

Grid search selected `learning_rate=0.1`, `n_estimators=300`, and `num_leaves=63` (CV accuracy: 0.527). The more expressive trees (63 leaves vs. the initial 31) help the model use the upweighted minority signal.

The model reaches 53.5% accuracy and macro F1 of 0.30. Per-class spread is reasonable: ratings 2/3/4 reach F1 of 0.05/0.07/0.13, rating 5 has recall 0.74, and rating 1 stays the strongest minority class (F1 = 0.52).

Despite the trigram vocabulary, every top-20 feature is a unigram ("good", "not", "love", "cant"). No bigrams or trigrams rank highly, suggesting the dataset is too small for longer n-grams to accumulate enough signal. This also explains why LightGBM does not outperform the linear models here: tree ensembles need denser, more interactive features to show their advantage, which a small TF-IDF matrix does not provide. The negative result from the expanded hyperparameter search supports this interpretation. If the bottleneck were regularization or tree shape, adding `reg_alpha`, `reg_lambda`, `min_child_samples`, and `colsample_bytree` to the search should have moved the metrics. We tried this implementation and it did not, which points to the representation itself as the limiting factor rather than the model's hyperparameters.

## 11. Model 4: Bidirectional LSTM

**Design choices.**
- **Text augmentation** via synonym replacement for underrepresented classes, augmenting up to the count of rating 1 rather than rating 5 to avoid generating excessive synthetic samples.
- **Class weights** (inversely proportional to class frequency) during training, on top of augmentation.
- **A small Bidirectional LSTM** (32 units) with aggressive dropout (0.3–0.4). A larger architecture would overfit severely on ~6,000 reviews, because the embedding layer alone has 1.28M parameters.
- **EarlyStopping** on validation loss with patience 3.

### Architecture

- **Embedding (64-dim)** - dense vector per word.
- **SpatialDropout1D (0.3)** - drops entire embedding channels, which regularizes sequence models better than standard dropout.
- **Bidirectional LSTM (32 units)** - reads context in both directions.
- **Dense (16, ReLU) + Dropout (0.4)** - small head with high dropout.
- **Dense (5, softmax)** - output over 5 rating classes.

### Results and discussion

The LSTM reaches 50.0% accuracy and macro F1 of 0.32. EarlyStopping restored weights from epoch 6 (best val_loss: 1.118), where the gap between training accuracy 0.66 and validation accuracy 0.61 is already visible. Overfitting begins from epoch 7 despite the regularization. This reflects the fundamental limitation of training a 1.28M-parameter network on ~6,000 reviews.

Per-class results show the most balanced spread of all four models: rating 5 at F1 = 0.72, rating 1 at 0.44, and ratings 2/3/4 at 0.15/0.15/0.14. Compared to the other models, the LSTM trades some rating 1 performance for more even coverage of the middle ratings. The combined effect of augmentation and class weighting redistributing the model's capacity.

We tested several variants (`MAX_LEN=150`, patience 5, augmenting up to rating 5's count) but macro F1 stayed at 0.32–0.33, so we kept the simpler configuration. This stability suggests the bottleneck is dataset size and architecture, not hyperparameters. We arrived at some likely improvements such as using pre-trained embeddings (GloVe, FastText), a pre-trained transformer (DistilBERT), or reducing to 3 classes (negative/neutral/positive) to merge the ambiguous middle ratings. However, we concluded that these implementations are either beyond the scope of this assignment or don't address the task effectively.

## 12. Model Comparison and Summary


| Model | Accuracy | Macro F1 | Notes |
|---|---|---|---|
| Logistic Regression | 0.494 | 0.33 | Class weighted; C=1.0, L2 penalty found via grid search |
| LinearSVC (SVM) | 0.601 | 0.35 | Best macro F1; C=0.1 found via 5-fold CV |
| LightGBM | 0.535 | 0.30 | Class weighted; best params found via grid search |
| Bidirectional LSTM | 0.500 | 0.32 | Most balanced minority class coverage |


**Key findings.** The SVM is the best overall (highest accuracy and macro F1). Linear models with TF-IDF are competitive, this is typical for small text datasets where TF-IDF already captures most of the signal and complex models cannot show their advantage. The LSTM has the most even spread across minority classes, but at the cost of rating 1 performance, so its macro F1 lands close to LR despite very different behavior. LightGBM is the weakest, reflecting that tree ensembles struggle to exploit sparse high-dimensional TF-IDF features when bigrams and trigrams contribute little signal.

**Why macro F1 matters here.** LR and the LSTM have similar macro F1 (0.33 vs. 0.32) despite quite different accuracy (0.494 vs. 0.500), because accuracy is dominated by rating 5 while macro F1 weights all classes equally. This confirms macro F1 as the more informative metric for this task.

**Class imbalance is the dominant constraint.** Across all four models, ratings 2 and 3 stay as the hardest classes to categorize (F1 = 0.05–0.15), even with class weighting and augmentation. The middle ratings are highly ambiguous in the data, many such reviews mix positive and negative language, and no amount of tuning recovers them.

**Overfitting control across models.** L2 regularization with tuned C (LR, SVM), tree complexity tuning (LightGBM), dropout and EarlyStopping (LSTM), plus the held-out test set and CV-based hyperparameter selection used everywhere.

---

# Task 2: AI vs. Human Art Classifier

**Dataset:** `Art_shuffled/` - 539 AI-generated, 436 real art images (975 total).
**Model:** EfficientNetB0 via transfer learning.

## 1. Data Analysis

The dataset has a mild 55/45 imbalance (AI/real). A naive classifier always predicting "AI" would reach ~55% accuracy, so meaningful performance must clearly exceed this. With fewer than 1,000 images, training a deep CNN from scratch would overfit severely, this motivated our choice of transfer learning.

## 2. Preprocessing and augmentation

All images are resized to 224×224 to match EfficientNetB0's expected input. To compensate for the small dataset, we apply light augmentation (rotation ±15°, horizontal flip, zoom ±10%) during training to increase variety and reduce overfitting. 20% of the data is held out for validation using `ImageDataGenerator`'s `subset` parameter, ensuring consistent splitting.

## 3. Architecture - EfficientNetB0

We use EfficientNetB0 pretrained on ImageNet. EfficientNet's compound scaling gives strong performance per parameter, B0 is the smallest variant (suitable for our computational constraint), and transfer learning leverages ImageNet features that transfer well to art images. With <1,000 training samples, a smaller pretrained model is preferable to a larger one, the additional capacity of B3 or B7 would not be supportable.

The custom head on top of the frozen base:

- **GlobalAveragePooling2D** - reduces feature maps to a single vector per channel, avoiding the very large parameter count that flattening would produce.
- **Dense(128, ReLU)** - task-specific feature combinations.
- **Dropout(0.3)** - regularization in the head.
- **Dense(1, sigmoid)** - binary output (0 = AI, 1 = real).

## 4. Training in two phases

**Phase 1 - frozen base.** Only the head is trained, so the new layers can adapt to the task without disrupting pretrained ImageNet features. Adam at `lr=1e-3` is appropriate since only the small head is updated. EarlyStopping on `val_loss` with patience 3 and `restore_best_weights=True`.

**Phase 2 - fine-tuning.** We unfreeze the last 20 layers of EfficientNetB0 to adapt higher-level features toward AI-vs-human cues (brushstroke regularity, noise distribution, stylistic consistency etc.). Earlier layers capture transferable low-level features and stay frozen. Adam at `lr=1e-5` - a much lower rate, since aggressive updates would destroy the pretrained representations. EarlyStopping again with patience 3.

## 5. Evaluation Metrics

## Summary and Discussion

### Results

The model achieves 75% validation accuracy on the held-out validation set (194 images). The best validation accuracy observed during training was 77.8%, from which EarlyStopping restored weights. The small gap between this and the final evaluation score reflects normal variation between the shuffled training generator and the fixed evaluation set. Given the small dataset size and the inherent difficulty of the task, 75% is a reasonable result.

### Class-wise performance

The classification report reveals an asymmetry between classes:
- AI Art (precision 74%, recall 84%) — the model is effective at detecting AI-generated images, but at the cost of misclassifying some real images as AI
- Real Art (precision 76%, recall 63%) — the model is more cautious here, missing 37% of genuine human-made images

This asymmetry is consistent with the mild class imbalance (55% AI, 45% real): the model learns a slight bias toward the majority class.

### Overfitting analysis

The training curves show a moderate overfitting pattern: training accuracy climbs to ~86% while validation accuracy plateaus around 77–79%. This gap is smaller than might be expected for a dataset of this size, which suggests the regularization measures were effective. EarlyStopping triggered early in both phases, after epoch 5 in Phase 1 and epoch 4 in Phase 2, preventing further divergence between training and validation performance.

Mitigation strategies applied:
- **Dropout (0.3)** in the classification head
- **Data augmentation** (rotation, horizontal flip, zoom)
- **EarlyStopping** with `restore_best_weights=True`

Further improvements that were not pursued due to resource constraints:
- More aggressive augmentation (color jitter, random cropping)
- L2 weight regularization in the dense layer
- A larger dataset would be the most impactful fix

### Fine-tuning assessment

Phase 2 fine-tuning produced only marginal gains over Phase 1, with EarlyStopping restoring weights from epoch 1 of Phase 2. Effectively the model improved very little during fine-tuning. This suggests the frozen ImageNet features were already well-suited to the task, and that the primary bottleneck is dataset size rather than feature quality.

## Task 2b: Five New Images

We test the trained model on five AI-generated images stored locally in `task2b_aiart_images/`. The true label for all five is `AiArtData` (class index 0). Each image is loaded as RGB, resized to 224×224, and passed through the model - EfficientNetB0 includes its own internal preprocessing, so no manual rescaling is needed. The sigmoid output is thresholded at 0.5 (below = AI, above = real).

### Task 2b — Results on new images

The model was tested on five images we sourced ourselves: three AI-generated (one screenshot of an AI-generated piece in a frame, one AI image of a man in a kitchen, and one stylized lion from an advanced image generator) and two real photographs (a landscape over Bergen and a desk scene with a water bottle). The breakdown:

| Image | True | Predicted | P(Real) | Confidence |
|-------|------|-----------|---------|------------|
| Framed AI artwork | AiArtData | AiArtData | 0.46 | 54.5% |
| AI man in kitchen | AiArtData | RealArt | 0.53 | 53.1% |
| Stylized lion (advanced generator) | AiArtData | AiArtData | 0.14 | 85.8% |
| Bergen landscape photo | RealArt | AiArtData | 0.47 | 53.0% |
| Desk with water bottle | RealArt | AiArtData | 0.32 | 68.0% |

**Accuracy: 2/5 (40%)** — well below the 76% validation accuracy. The model correctly classified two of the three AI images and got both real photographs wrong, calling them AI-generated. There are several patterns that are worth noting:

**Confidence is highest where it should be.** The only confident, correct prediction (85.8%) is the stylized lion, an image with the saturated colors and clean illustrative style that dominate the AI training set. The other correct prediction (the framed AI artwork) sits just barely on the right side of the 0.5 threshold at 54.5% confidence, which is essentially a coin flip.

**The "advanced generator" miss is informative.** The AI-generated kitchen scene was misclassified, and crucially this was an image from a recent, high-quality generator that produces photorealistic output rather than illustrative output. When confronted with a photorealistic AI image, the model, mostly trained on illustrative AI art, has no learned cues to fall back on.

**The model is biased toward predicting "AI".** Both real photographs were misclassified, and this is consistent with the bias we observed on the validation set (Real Art recall was only 0.64).

### Consequences and limitations

A 76% classifier misclassifies roughly 1 in 4 images, but our task 2b results suggest this number is optimistic for real-world use. On five out-of-distribution images we sourced ourselves, accuracy dropped to 40%, with both casual photographs misclassified as AI and a photorealistic AI image misclassified as real.

For applications with real-world stakes: art attribution, copyright proceedings, gallery curation, this is too unreliable to use as a definitive classifier. It should be treated as a screening or supporting tool that flags candidates for human review, and only on images stylistically similar to the training distribution.
---

## Use of AI Tools

We used AI assistants (ChatGPT and Claude) primarily as a productivity aid for routine implementation work, so we could focus on the conceptually important parts of the assignment: model selection, evaluation strategy, and interpretation of results. Specifically:

- **Data cleaning implementation** - writing regex patterns for detecting gibberish, spam repetition, emoji spam, and non-English text.
- **Boilerplate EDA code** - formatting summary statistics, generating standard distribution plots, and print-statement scaffolding for inspecting flagged reviews and model scores.
- **Plotting and reporting** - matplotlib/seaborn boilerplate for confusion matrices, training curves, and feature-importance plots.
